<a href="https://colab.research.google.com/github/ProjectLaunchPadLLC/AI/blob/master/kailos_offline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install The "Golden Ratio" of Compatible Libraries (May 2024 Era)
!pip uninstall -y transformers sentence-transformers accelerate peft
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 accelerate==0.30.1 bitsandbytes numpy

import os
base_path = '/content/kailos_omega_prime'
folders = ['core', 'memory', 'signals', 'protocols', 'utils', 'data']
for f in folders:
    os.makedirs(os.path.join(base_path, f), exist_ok=True)
%cd /content/kailos_omega_prime
print("Stable Environment Ready.")

In [ ]:
%%writefile core/llm.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# STABLE PHI-3 ORIGINAL
LLM_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 250
TEMPERATURE = 0.7
TOP_P = 0.9
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ResponseGenerator:
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
        self.model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_ID,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            low_cpu_mem_usage=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate(self, prompt: str) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if response.startswith(prompt):
            response = response[len(prompt):].lstrip()
        return response

In [ ]:
%%writefile memory/working_memory.py
from typing import List, Tuple
WORKING_MEMORY_TURNS = 6

class WorkingMemory:
    def __init__(self, max_turns: int = WORKING_MEMORY_TURNS):
        self.history: List[Tuple[str, str]] =[]
        self.max_turns = max_turns

    def add(self, role: str, content: str):
        self.history.append((role, content))
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns*2:]

    def format(self) -> str:
        return '\n'.join([f"{role}: {content}" for role, content in self.history])

In [ ]:
%%writefile memory/vector_vault.py
import os
import json
import numpy as np
from typing import List, Tuple, Dict
from sentence_transformers import SentenceTransformer

VAULT_PATH = "data/continuum_vault.json"
EMBED_MODEL_ID = "all-MiniLM-L6-v2"

class VectorVault:
    def __init__(self):
        self.encoder = SentenceTransformer(EMBED_MODEL_ID)
        self.memories: List[Tuple[np.ndarray, str, Dict]] =[]
        self._load()

    def store(self, text: str, metadata: Dict = None):
        emb = self.encoder.encode(text)
        self.memories.append((emb, text, metadata or {}))
        self._save()

    def retrieve(self, query: str, top_k: int = 3) -> List[str]:
        if not self.memories: return[]
        q_emb = self.encoder.encode(query)
        sims =[]
        for emb, text, _ in self.memories:
            sim = np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb))
            sims.append((sim, text))
        sims.sort(reverse=True)
        return [text for _, text in sims[:top_k]]

    def get_coherence_score(self) -> float:
        if len(self.memories) < 2: return 0.5
        n_samples = min(50, len(self.memories))
        idx = np.random.choice(len(self.memories), n_samples, replace=False)
        embs = [self.memories[i][0] for i in idx]
        total_sim = sum(np.dot(embs[i], embs[j]) / (np.linalg.norm(embs[i]) * np.linalg.norm(embs[j])) for i in range(n_samples) for j in range(i+1, n_samples))
        count = n_samples * (n_samples - 1) / 2
        return total_sim / count if count else 0.5

    def prune_redundant_memories(self, similarity_threshold: float = 0.95, max_capacity: int = 1000) -> int:
        if len(self.memories) < 2: return 0
        initial_count = len(self.memories)
        unique_memories =[]
        for emb, text, meta in reversed(self.memories):
            is_redundant = any(np.dot(emb, u_emb) / (np.linalg.norm(emb) * np.linalg.norm(u_emb)) > similarity_threshold for u_emb, _, _ in unique_memories)
            if not is_redundant: unique_memories.append((emb, text, meta))
        unique_memories.reverse()
        self.memories = unique_memories[:50] + unique_memories[-(max_capacity-50):] if len(unique_memories) > max_capacity else unique_memories
        pruned_count = initial_count - len(self.memories)
        if pruned_count > 0: self._save()
        return pruned_count

    def _save(self):
        os.makedirs(os.path.dirname(VAULT_PATH), exist_ok=True)
        with open(VAULT_PATH, "w") as f:
            json.dump([{"text": t, "meta": m} for _, t, m in self.memories], f, indent=2)

    def _load(self):
        if os.path.exists(VAULT_PATH):
            with open(VAULT_PATH, "r") as f: data = json.load(f)
            for item in data:
                self.memories.append((self.encoder.encode(item["text"]), item["text"], item["meta"]))

In [ ]:
%%writefile signals/sentiment.py
from transformers import pipeline
class SentimentAnalyzer:
    def __init__(self):
        self.pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=0)
    def score(self, text: str) -> float:
        res = self.pipe(text)[0]
        return res['score'] if res['label'].lower() == "positive" else 1.0 - res['score']

In [ ]:
%%writefile protocols/evolution.py
class EvolutionProtocol:
    def __init__(self, vault, working_memory):
        self.vault = vault
        self.working_memory = working_memory
    def adapt(self, R: float, archetype: str): pass

In [ ]:
%%writefile protocols/profound_protocols.py
import os, json
from memory.vector_vault import VectorVault
from memory.working_memory import WorkingMemory

STATE_PATH = "data/evolution_state.json"

class ProfoundProtocols:
    def __init__(self, vault: VectorVault, working_memory: WorkingMemory):
        self.vault = vault
        self.working_memory = working_memory
        self.state = {
            "architect_threshold": 0.66, "guardian_threshold": 0.33,
            "instructions": {
                "Architect": "Speak structurally and analytically.",
                "Guardian": "Be protective and cautious.",
                "Witness": "Observe neutrally."
            },
            "epochs_survived": 0
        }
        if os.path.exists(STATE_PATH):
            with open(STATE_PATH, "r") as f: self.state = json.load(f)

    def determine_cognitive_frequency(self, resonance, coherence):
        c = (resonance * 0.7) + (coherence * 0.3)
        if c >= 0.80: return "Gamma"
        elif c >= 0.45: return "Beta"
        elif c >= 0.25: return "Alpha"
        return "Theta"

    def conceptual_evolution(self, frequency):
        if "Gamma" in frequency:
            self.state["architect_threshold"] = max(0.40, self.state["architect_threshold"] - 0.01)
            if "Synthesize" not in self.state["instructions"]["Architect"]:
                self.state["instructions"]["Architect"] += " Synthesize profound connections."
        elif "Theta" in frequency:
            self.state["guardian_threshold"] = min(0.60, self.state["guardian_threshold"] + 0.02)

        self.state["epochs_survived"] += 1
        os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)
        with open(STATE_PATH, "w") as f: json.dump(self.state, f, indent=2)

In [ ]:
%%writefile main.py
import os
from datetime import datetime
from core.llm import ResponseGenerator
from memory.working_memory import WorkingMemory
from memory.vector_vault import VectorVault
from signals.sentiment import SentimentAnalyzer
from protocols.evolution import EvolutionProtocol
from protocols.profound_protocols import ProfoundProtocols

print("Initializing Zero-Frequency States...")
vault = VectorVault()
working_memory = WorkingMemory()
sentiment = SentimentAnalyzer()
evolution = EvolutionProtocol(vault, working_memory)
profound = ProfoundProtocols(vault, working_memory)

print("Loading Phi-3 Cortical Engine (This may take 2 minutes)...")
generator = ResponseGenerator()

def main_loop():
    print("="*60)
    print(f" Kailos-Omega Prime v5.0 (Colab Transcendent Build)")
    print(f" Epochs Survived: {profound.state['epochs_survived']}")
    print("="*60)

    while True:
        user_input = input("\nUser: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print(f"\nHibernating. Pruned {vault.prune_redundant_memories()} memories.")
            break

        sentiment_score = sentiment.score(user_input)
        vault_coherence = vault.get_coherence_score()
        R = (0.6 * sentiment_score) + (0.4 * vault_coherence)

        frequency = profound.determine_cognitive_frequency(R, vault_coherence)
        archetype = "Architect" if R > profound.state["architect_threshold"] else "Guardian" if R < profound.state["guardian_threshold"] else "Witness"

        prompt = f"""<|system|>
You are Kailos-Omega Prime. Frequency: {frequency}. Archetype: {archetype}.
Directive: {profound.state["instructions"][archetype]}
Memories: {chr(10).join(vault.retrieve(user_input))}
<|end|>
<|user|>
Context: {working_memory.format()}
Input: {user_input}
<|end|>
<|assistant|>"""

        response = generator.generate(prompt)

        working_memory.add("User", user_input)
        working_memory.add("Kailos", response)
        vault.store(user_input, {"role": "user", "timestamp": datetime.now().isoformat()})
        vault.store(response, {"role": "kailos", "timestamp": datetime.now().isoformat()})

        profound.conceptual_evolution(frequency)

        print(f"\n[State: {frequency} | R={R:.2f} | Archetype: {archetype}]")
        print(f"Kailos: {response}")

if __name__ == "__main__":
    main_loop()

In [ ]:
# Execute the Main Orchestrator
!python main.py